# 01 — Baseline: can a pretrained YOLOv8 detect facade materials?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarEAbdelaal/M4U3-CV-Facade-Materials/blob/main/notebooks/01_baseline_inference.ipynb)

**Question.** Before collecting and labelling data, check whether an off-the-shelf model already solves the problem.

**Method.** Run YOLOv8s pretrained on COCO (80 everyday object classes) on 4 facade photos from the project dataset, one each for `brick`, `glass`, `cladding_panel` and `stone_cladding`, and compare its output with our labels.

**Expected answer.** COCO has no material classes, so the model can only report objects (people, cars, etc.), never surfaces. That is the reason for training a custom model in `02_training_eval.ipynb`.

Runs with no credentials: the images come from the project's GitHub Release, checked by SHA256. A GPU is not required.

In [ ]:
# Cell 1 — install (same pinned version as the training notebook)
%pip install -q ultralytics==8.4.163

In [ ]:
# Cell 2 — download the frozen dataset from the GitHub Release (course standard cell)
import os, hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/OmarEAbdelaal/M4U3-CV-Facade-Materials/releases/download/v1.0/facade-materials-v3-yolov8.zip"
SHA256 = "134e41be8c310452bbe8ea33fd55a891c6e29ca863f3c9b8df3827f59bb951cd"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]

already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)

In [ ]:
# Cell 3 — what can the pretrained model recognise?
from ultralytics import YOLO

coco_model = YOLO("yolov8s.pt")
coco_names = list(coco_model.names.values())
our_names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][k] for k in sorted(cfg["names"])]

print(f"COCO classes ({len(coco_names)}):")
print(", ".join(coco_names))
print()
overlap = [n for n in our_names if n.replace("_", " ") in coco_names or n in coco_names]
print("Our material classes:", our_names)
print("Found in COCO:", overlap if overlap else "none")

In [ ]:
# Cell 4 — run the pretrained model on one facade per material
import pandas as pd
from PIL import Image

OUT = Path("/content/results/baseline")
OUT.mkdir(parents=True, exist_ok=True)
CONF = 0.25
SAMPLE_CLASSES = ["brick", "glass", "cladding_panel", "stone_cladding"]

val_images = sorted((dataset_root / "valid" / "images").glob("*.jpg"))
samples = {c: next(p for p in val_images if p.name.startswith(c + "_")) for c in SAMPLE_CLASSES}

rows, preds = [], {}
for material, img in samples.items():
    r = coco_model.predict(str(img), conf=CONF, verbose=False)[0]
    preds[material] = r
    found = [f"{coco_names[int(c)]} ({s:.2f})" for c, s in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist())]
    rows.append({"image": img.name[:30], "true material": material,
                 "COCO detections": ", ".join(found) if found else "nothing",
                 "material detected": "no"})
    Image.fromarray(r.plot()[:, :, ::-1]).save(OUT / f"{material}_coco_pred.jpg")

table = pd.DataFrame(rows)
table.to_csv(OUT / "baseline_detections.csv", index=False)
table

In [ ]:
# Cell 5 — side by side: pretrained model output vs. our material labels
import matplotlib.pyplot as plt
from PIL import ImageDraw

def draw_labels(img_path):
    """Draw our ground-truth labels (boxes, or polygons converted to boxes)."""
    im = Image.open(img_path).convert("RGB")
    W, H = im.size
    d = ImageDraw.Draw(im)
    lbl = img_path.parent.parent / "labels" / (img_path.stem + ".txt")
    for line in lbl.read_text().splitlines():
        p = line.split()
        if not p:
            continue
        if len(p) > 5:
            xs, ys = [float(v) for v in p[1::2]], [float(v) for v in p[2::2]]
            x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
        else:
            cx, cy, w, h = map(float, p[1:5])
            x1, y1, x2, y2 = cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2
        d.rectangle([x1 * W, y1 * H, x2 * W, y2 * H], outline=(255, 200, 0), width=2)
    return im

fig, axes = plt.subplots(len(samples), 2, figsize=(10, 4.6 * len(samples)))
for row, (material, img) in enumerate(samples.items()):
    axes[row, 0].imshow(preds[material].plot()[:, :, ::-1])
    n = len(preds[material].boxes)
    axes[row, 0].set_title(f"Pretrained COCO model: {n} object(s), no material", fontsize=10)
    axes[row, 1].imshow(draw_labels(img))
    axes[row, 1].set_title(f"Our labels: {material}", fontsize=10)
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "baseline_comparison.jpg", dpi=110)
plt.show()
print("Saved to", OUT)

## Conclusion

- COCO's 80 classes are objects such as `person`, `car`, `bench` or `potted plant`. None of our 6 materials is among them, so the pretrained model **cannot output a material by design**: whatever it detects on a facade is an object in front of it, not the surface.
- The detections table (cell 4) and `results/baseline/baseline_comparison.jpg` show this on real facade photos.
- This mirrors Session 1: a generic model can be technically good and still unsuitable for the business question. Answering "which material is this facade?" needs our own classes, our own labelled data and a fine-tuned model, which is what `02_training_eval.ipynb` does.
- The pretrained weights are still useful: the custom model starts from them (transfer learning) rather than from zero.